<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/KREA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt
!apt install aria2 -qqy

# Your Krea 2 realism model
!mkdir -p /content/ComfyUI/models/diffusion_models
!wget -q --show-progress -O /content/ComfyUI/models/diffusion_models/Krea2Realism_V1.safetensors \
  "https://civitai.red/api/download/models/3405129?fileId=3296411&token=777a5632707c15ecf54e13eb3b1b862b"

# Text encoder
!mkdir -p /content/ComfyUI/models/text_encoders
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors \
  -d /content/ComfyUI/models/text_encoders -o qwen3vl_4b_fp8_scaled.safetensors

# VAE
!mkdir -p /content/ComfyUI/models/vae
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

!ls -lh /content/ComfyUI/models/diffusion_models/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/

/content
fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
t/ComfyUI/models/di   2%[                    ] 280.33M  79.3MB/s    eta 2m 25s 

  0%|          | 0/25 [00:00<?, ?it/s]

ealism_V1.safetenso  47%[========>           ]   5.87G  91.1MB/s    eta 98s    

  0%|          | 0/25 [00:00<?, ?it/s]

/content/ComfyUI/mo 100%[===================>]  12.24G  68.3MB/s    in 3m 22s  
 *** Download Progress Summary as of Wed Nov 11 01:15:44 2026 *** 
=
[#797153 3.6GiB/4.8GiB(74%) CN:16 DL:81MiB ETA:15s]
FILE: /content/ComfyUI/models/text_encoders/qwen3vl_4b_fp8_scaled.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
797153|OK  |    57MiB/s|/content/ComfyUI/models/text_encoders/qwen3vl_4b_fp8_scaled.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
f5d14e|OK  |   236MiB/s|/content/ComfyUI/models/vae/qwen_image_vae.safetensors

Status Legend:
(OK):download completed.
total 13G
-rw-r--r-- 1 root root 13G Oct 14 15:36 Krea2Realism_V1.safetensors
-rw-r--r-- 1 root root   0 Nov 11 00:37 put_diffusion_model_files_here
total 4.9G
-rw-r--r-- 1 root root    0 Nov 11 00

In [2]:
%cd /content/ComfyUI

import torch
import logging
import warnings
logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()

with torch.inference_mode():
    unet = UNETLoader.load_unet("Krea2Realism_V1.safetensors", "default")[0]
    clip = CLIPLoader.load_clip("qwen3vl_4b_fp8_scaled.safetensors", type="krea2")[0]
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]

print("Krea 2 loaded.")

/content/ComfyUI
Krea 2 loaded.


In [ ]:
%cd /content/ComfyUI

import torch
import random
from PIL import Image
import numpy as np
from IPython.display import display
from nodes import NODE_CLASS_MAPPINGS

# Define all nodes
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()

# Load models
with torch.inference_mode():
    unet = UNETLoader.load_unet("Krea2Realism_V1.safetensors", "default")[0]
    clip = CLIPLoader.load_clip("qwen3vl_4b_fp8_scaled.safetensors", type="krea2")[0]
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]

    positive = "photorealistic portrait of a woman, natural skin texture, detailed eyes, soft lighting, high quality"
    negative = ""

    pos = CLIPTextEncode.encode(clip, positive)[0]
    neg = ConditioningZeroOut.zero_out(pos)[0]
    latent = EmptyLatentImage.generate(1024, 1024, 1)[0]
    samples = KSampler.sample(
        unet, random.randint(1, 2**32 - 1), 8, 1.0, "euler", "simple",
        pos, neg, latent, 1.0
    )[0]
    image = VAEDecode.decode(vae, samples)[0]
    img = Image.fromarray((image.cpu().numpy().clip(0, 1) * 255).astype("uint8"))
    display(img)

/content/ComfyUI


In [1]:
# ========== Krea 2 API + Cloudflare Tunnel ==========
%cd /content/ComfyUI

!pip install -q fastapi uvicorn python-multipart requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Optional
import base64
from io import BytesIO
import subprocess
import time
import re
import threading
import torch
import random
from PIL import Image
import numpy as np
from nodes import NODE_CLASS_MAPPINGS

# Nodes
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()
LoraLoaderModelOnly = NODE_CLASS_MAPPINGS["LoraLoaderModelOnly"]()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["GET", "POST", "OPTIONS", "PUT", "DELETE"],
    allow_headers=["*"],
    expose_headers=["*"],
)

@app.middleware("http")
async def add_cors_headers(request: Request, call_next):
    if request.method == "OPTIONS":
        return JSONResponse(
            content={},
            headers={
                "Access-Control-Allow-Origin": "*",
                "Access-Control-Allow-Methods": "GET, POST, OPTIONS, PUT, DELETE",
                "Access-Control-Allow-Headers": "*",
            }
        )
    response = await call_next(request)
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS, PUT, DELETE"
    response.headers["Access-Control-Allow-Headers"] = "*"
    return response

print("Loading Krea 2 models (once)...")
with torch.inference_mode():
    unet = UNETLoader.load_unet("Krea2Realism_V1.safetensors", "default")[0]
    clip = CLIPLoader.load_clip("qwen3vl_4b_fp8_scaled.safetensors", type="krea2")[0]
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]
print("Models loaded.")

class LoraItem(BaseModel):
    name: str
    strength: float = 1.0

class GenerateRequest(BaseModel):
    checkpoint: Optional[str] = None
    sampler: str = "euler"
    scheduler: str = "simple"
    steps: int = 8
    cfg: float = 1.0
    width: int = 1024
    height: int = 1024
    loras: List[LoraItem] = []
    positive: str
    negative: str = ""

@app.post("/generate")
async def generate(req: GenerateRequest):
    try:
        with torch.inference_mode():
            current_model = unet

            for lora_item in req.loras:
                if lora_item.name and lora_item.strength > 0:
                    current_model = LoraLoaderModelOnly.load_lora_model_only(
                        current_model, lora_item.name, float(lora_item.strength)
                    )[0]

            positive = CLIPTextEncode.encode(clip, req.positive)[0]
            negative = ConditioningZeroOut.zero_out(positive)[0] if not req.negative.strip() else CLIPTextEncode.encode(clip, req.negative)[0]
            latent_image = EmptyLatentImage.generate(req.width, req.height, 1)[0]

            seed = random.randint(0, 2**32 - 1)
            samples = KSampler.sample(
                current_model, seed, req.steps, req.cfg,
                req.sampler, req.scheduler,
                positive, negative, latent_image, 1.0
            )[0]

            decoded = VAEDecode.decode(vae, samples)[0].detach()
            img_array = (decoded.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)[0]
            image = Image.fromarray(img_array)

            buffer = BytesIO()
            image.save(buffer, format="PNG")
            img_b64 = base64.b64encode(buffer.getvalue()).decode()

            return {"image": f"data:image/png;base64,{img_b64}", "seed": seed}
    except Exception as e:
        return {"error": str(e)}

@app.get("/")
def root():
    return {"status": "ok", "model": "Krea2Realism_V1"}

@app.get("/user_models")
def get_user_models():
    return {"checkpoints": [], "loras": []}

def start_server():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

!fuser -k 8000/tcp || true
!pkill -f cloudflared || true
time.sleep(2)

threading.Thread(target=start_server, daemon=True).start()
print("Waiting for server to start...")
time.sleep(6)

print("Starting Cloudflare tunnel...")
proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(50):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*60)
    print("BACKEND READY")
    print(f"Paste this into the frontend Backend URL field:\n{public_url}")
    print("="*60)
else:
    print("Tunnel failed.")

/content/ComfyUI
Loading Krea 2 models (once)...
Models loaded.
^C
Waiting for server to start...
Starting Cloudflare tunnel...
2026-11-11T01:47:50Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-11T01:47:50Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-11T01:47:55Z INF +--------------------------------------------------------------------------------------------+
2026-11-11T01:47:55Z INF |  Your quick Tunnel has been created! Visit it 